In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
import random
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings('ignore')


SEEDS = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555]  
NUM_SEEDS = len(SEEDS)

def set_seed(seed):
    """Set seed for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")



def load_and_prepare_reasonable_dataset(file_path, test_size=0.2, min_samples_per_author=50, max_authors=1000, seed=42):
    """
    Load Google Jam dataset with reasonable parameters
    Args:
        file_path: Path to CSV file
        test_size: Proportion of data for testing
        min_samples_per_author: Minimum samples per author to include
        max_authors: Maximum number of authors to include (top N by sample count)
        seed: Random seed for reproducibility
    """
    print(f"Loading dataset with seed {seed}...")
    data = pd.read_csv(file_path)
    
    print(f"Dataset shape: {data.shape}")
    

    if 'username' not in data.columns or 'flines' not in data.columns:
        raise ValueError("Dataset must contain 'username' and 'flines' columns")
    
    
    data = data.dropna(subset=['flines', 'username'])
    data['flines'] = data['flines'].astype(str)
    data = data[data['flines'].str.strip() != '']
    
  
    author_counts = data['username'].value_counts()
    print(f"\nTotal unique authors: {len(author_counts)}")
    print(f"Total samples: {len(data)}")
    
   
    valid_authors = author_counts[author_counts >= min_samples_per_author].index
    filtered_data = data[data['username'].isin(valid_authors)]
    
    print(f"\nAfter filtering (minimum {min_samples_per_author} samples per author):")
    print(f"Number of authors: {len(valid_authors)}")
    print(f"Total samples: {len(filtered_data)}")
    
    
    if len(valid_authors) > max_authors:
        top_authors = author_counts.head(max_authors).index
        filtered_data = filtered_data[filtered_data['username'].isin(top_authors)]
        print(f"\nTaking top {max_authors} authors by sample count:")
        print(f"Number of authors: {len(top_authors)}")
        print(f"Total samples: {len(filtered_data)}")
    
    
    label_encoder = LabelEncoder()
    filtered_data['EncodedLabels'] = label_encoder.fit_transform(filtered_data['username'])
    num_classes = len(label_encoder.classes_)
    
    print(f"\nNumber of classes: {num_classes}")
    
    train_data, test_data = custom_stratified_split(filtered_data, test_size=test_size, seed=seed)
    
    print(f"\nTrain set size: {len(train_data)}")
    print(f"Test set size: {len(test_data)}")
    
    return train_data, test_data, label_encoder, num_classes

def custom_stratified_split(data, test_size=0.2, seed=42):
    """
    Custom stratified split that ensures proper distribution
    """
    train_data = []
    test_data = []
    
    np.random.seed(seed)
    
    grouped = data.groupby('username')
    
    for author, group in grouped:
        group = group.sample(frac=1, random_state=seed).reset_index(drop=True)
        
        n_test = max(1, int(len(group) * test_size))
        
        if len(group) - n_test < 2:
            n_test = max(1, len(group) - 2)
        
        test_samples = group.iloc[:n_test]
        train_samples = group.iloc[n_test:]
        
        test_data.append(test_samples)
        train_data.append(train_samples)
    
    train_data = pd.concat(train_data, ignore_index=True)
    test_data = pd.concat(test_data, ignore_index=True)
    
    train_data = train_data.sample(frac=1, random_state=seed).reset_index(drop=True)
    test_data = test_data.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return train_data, test_data


tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

def tokenize_code(code, max_length=256):
    return tokenizer(
        code,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

class CodeDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokens = tokenize_code(row["flines"])
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["EncodedLabels"], dtype=torch.long)
        }


class CodeBERTStage1(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.codebert = AutoModel.from_pretrained("microsoft/codebert-base")
        self.dropout = nn.Dropout(0.3)  
        self.classifier = nn.Linear(
            self.codebert.config.hidden_size,
            num_classes
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        token_embeddings = outputs.last_hidden_state
        attention_mask_exp = attention_mask.unsqueeze(-1)

        pooled_output = (token_embeddings * attention_mask_exp).sum(dim=1) / attention_mask_exp.sum(dim=1)
        pooled_output = self.dropout(pooled_output)  

        logits = self.classifier(pooled_output)
        return logits


class CodeBERT_RI_Transformer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.codebert = AutoModel.from_pretrained(
            "microsoft/codebert-base",
            output_hidden_states=True
        )

        hidden = self.codebert.config.hidden_size

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=8,
            batch_first=True
        )

        self.layer_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.att_fc = nn.Linear(hidden, hidden)
        self.context_vector = nn.Parameter(torch.randn(hidden))

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden_states = outputs.hidden_states[1:]  
        attention_mask_exp = attention_mask.unsqueeze(-1)

        layerwise_embeddings = []

        for layer in hidden_states:
            pooled = (layer * attention_mask_exp).sum(dim=1) / attention_mask_exp.sum(dim=1)
            layerwise_embeddings.append(pooled)

        layer_sequence = torch.stack(layerwise_embeddings, dim=1)

        h = self.layer_transformer(layer_sequence)

        u = torch.tanh(self.att_fc(h))
        scores = torch.matmul(u, self.context_vector)
        alpha = torch.softmax(scores, dim=1)

        x_out = torch.sum(h * alpha.unsqueeze(-1), dim=1)

        return self.classifier(x_out)


def run_single_experiment(seed, file_path, min_samples=50, max_authors=100, 
                         epochs_stage1=3, epochs_stage2=3):
    """
    Run a single experiment with a given seed
    Returns: (stage1_accuracy, stage2_accuracy, improvement)
    """
    print(f"\n{'='*70}")
    print(f"EXPERIMENT WITH SEED: {seed}")
    print(f"{'='*70}")
    
    set_seed(seed)
    
    train_data, test_data, label_encoder, num_classes = load_and_prepare_reasonable_dataset(
        file_path, 
        test_size=0.2,
        min_samples_per_author=min_samples,
        max_authors=max_authors,
        seed=seed
    )
    
    train_dataset = CodeDataset(train_data)
    test_dataset = CodeDataset(test_data)
    
    batch_size = 8
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    
    print("\n" + "="*50)
    print("STAGE 1: Training Basic CodeBERT Model")
    print("="*50)
    
    stage1_model = CodeBERTStage1(num_classes).to(device)
    
    optimizer = optim.AdamW(stage1_model.parameters(), lr=2e-5, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    
    stage1_model.train()
    
    for epoch in range(epochs_stage1):
        total_loss = 0
        correct = 0
        total = 0
        
        for step, batch in enumerate(train_loader, 1):
            optimizer.zero_grad()

            logits = stage1_model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )

            loss = criterion(
                logits,
                batch["labels"].to(device)
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(stage1_model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            batch_total = batch["labels"].size(0)
            batch_correct = (predicted.cpu() == batch["labels"]).sum().item()
            total += batch_total
            correct += batch_correct
        
        epoch_acc = 100 * correct / total
        avg_loss = total_loss / len(train_loader)
        print(f"[Stage 1] Epoch {epoch+1} Loss: {avg_loss:.4f}, Acc: {epoch_acc:.2f}%")
    
    torch.save(
        stage1_model.codebert.state_dict(),
        f"/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/codebert_stage1_weights_seed{seed}.pt"
    )
    
    
    print("\n" + "="*50)
    print("STAGE 2: Training Enhanced CodeBERT with Transformer Layer")
    print("="*50)
    
    stage2_model = CodeBERT_RI_Transformer(num_classes).to(device)
    
    stage2_model.codebert.load_state_dict(
        torch.load(f"/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/codebert_stage1_weights_seed{seed}.pt")
    )
    
    for param in stage2_model.codebert.parameters():
        param.requires_grad = False
    
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, stage2_model.parameters()),
        lr=1e-4,
        weight_decay=0.01
    )
    
    stage2_model.train()
    
    for epoch in range(epochs_stage2):
        total_loss = 0
        correct = 0
        total = 0
        
        for step, batch in enumerate(train_loader, 1):
            optimizer.zero_grad()

            logits = stage2_model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )

            loss = criterion(
                logits,
                batch["labels"].to(device)
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, stage2_model.parameters()), 
                max_norm=1.0
            )
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            batch_total = batch["labels"].size(0)
            batch_correct = (predicted.cpu() == batch["labels"]).sum().item()
            total += batch_total
            correct += batch_correct
        
        epoch_acc = 100 * correct / total
        avg_loss = total_loss / len(train_loader)
        print(f"[Stage 2] Epoch {epoch+1} Loss: {avg_loss:.4f}, Acc: {epoch_acc:.2f}%")
    
    torch.save(
        stage2_model.state_dict(),
        f"/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/codebert_full_RI_transformer_seed{seed}.pt"
    )
    
    
    print("\n" + "="*50)
    print("EVALUATION ON TEST SET")
    print("="*50)
    
    stage1_model.eval()
    all_preds_stage1, all_labels = [], []
    
    with torch.no_grad():
        for batch in test_loader:
            logits = stage1_model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            labels = batch["labels"].cpu().numpy()
            all_preds_stage1.extend(preds)
            all_labels.extend(labels)
    
    stage1_accuracy = accuracy_score(all_labels, all_preds_stage1)
    
    stage2_model.eval()
    all_preds_stage2 = []
    
    with torch.no_grad():
        for batch in test_loader:
            logits = stage2_model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds_stage2.extend(preds)
    
    stage2_accuracy = accuracy_score(all_labels, all_preds_stage2)
    improvement = stage2_accuracy - stage1_accuracy
    
    print(f"\nResults for seed {seed}:")
    print(f"  Stage 1 Accuracy: {stage1_accuracy:.4f}")
    print(f"  Stage 2 Accuracy: {stage2_accuracy:.4f}")
    print(f"  Improvement: {improvement:.4f}")
    if stage1_accuracy > 0:
        print(f"  Improvement %: {improvement/stage1_accuracy*100:.2f}%")
    
    del stage1_model, stage2_model
    torch.cuda.empty_cache()
    
    return stage1_accuracy, stage2_accuracy, improvement


def perform_statistical_analysis(results_df):
    """
    Perform statistical significance testing on the results
    """
    print("\n" + "="*70)
    print("STATISTICAL SIGNIFICANCE ANALYSIS")
    print("="*70)
    
    stage1_accuracies = results_df['Stage1_Accuracy'].values
    stage2_accuracies = results_df['Stage2_Accuracy'].values
    improvements = results_df['Improvement'].values
    
    print("\n1. DESCRIPTIVE STATISTICS:")
    print(f"Stage 1 Accuracy:")
    print(f"  Mean: {np.mean(stage1_accuracies):.4f}")
    print(f"  Std: {np.std(stage1_accuracies):.4f}")
    print(f"  Min: {np.min(stage1_accuracies):.4f}")
    print(f"  Max: {np.max(stage1_accuracies):.4f}")
    
    print(f"\nStage 2 Accuracy:")
    print(f"  Mean: {np.mean(stage2_accuracies):.4f}")
    print(f"  Std: {np.std(stage2_accuracies):.4f}")
    print(f"  Min: {np.min(stage2_accuracies):.4f}")
    print(f"  Max: {np.max(stage2_accuracies):.4f}")
    
    print(f"\nImprovement (Stage2 - Stage1):")
    print(f"  Mean: {np.mean(improvements):.4f}")
    print(f"  Std: {np.std(improvements):.4f}")
    print(f"  Min: {np.min(improvements):.4f}")
    print(f"  Max: {np.max(improvements):.4f}")
    
    print("\n2. PAIRED T-TEST (Stage 1 vs Stage 2):")
    t_stat, p_value = stats.ttest_rel(stage2_accuracies, stage1_accuracies)
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    
    if p_value < 0.05:
        print(f"  Result: SIGNIFICANT difference (p < 0.05)")
        if np.mean(improvements) > 0:
            print(f"  Conclusion: Stage 2 is SIGNIFICANTLY BETTER than Stage 1")
        else:
            print(f"  Conclusion: Stage 2 is SIGNIFICANTLY WORSE than Stage 1")
    else:
        print(f"  Result: NO SIGNIFICANT difference (p >= 0.05)")
    
    print("\n3. 95% CONFIDENCE INTERVAL FOR IMPROVEMENT:")
    ci_low, ci_high = stats.t.interval(
        confidence=0.95,
        df=len(improvements)-1,
        loc=np.mean(improvements),
        scale=stats.sem(improvements)
    )
    print(f"  95% CI: [{ci_low:.4f}, {ci_high:.4f}]")
    
    if ci_low > 0:
        print("  CI interpretation: Improvement is SIGNIFICANTLY positive (CI doesn't contain 0)")
    elif ci_high < 0:
        print("  CI interpretation: Improvement is SIGNIFICANTLY negative (CI doesn't contain 0)")
    else:
        print("  CI interpretation: Improvement is NOT SIGNIFICANT (CI contains 0)")
    
    print("\n4. WILCOXON SIGNED-RANK TEST (non-parametric):")
    w_stat, p_value_wilcoxon = stats.wilcoxon(stage2_accuracies, stage1_accuracies)
    print(f"  Wilcoxon statistic: {w_stat:.4f}")
    print(f"  p-value: {p_value_wilcoxon:.6f}")
    
    print("\n5. EFFECT SIZE (Cohen's d):")
    n1, n2 = len(stage1_accuracies), len(stage2_accuracies)
    s1, s2 = np.std(stage1_accuracies, ddof=1), np.std(stage2_accuracies, ddof=1)
    pooled_std = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1 + n2 - 2))
    cohens_d = np.mean(improvements) / pooled_std
    print(f"  Cohen's d: {cohens_d:.4f}")
    
    if abs(cohens_d) < 0.2:
        effect_size = "Negligible"
    elif abs(cohens_d) < 0.5:
        effect_size = "Small"
    elif abs(cohens_d) < 0.8:
        effect_size = "Medium"
    else:
        effect_size = "Large"
    print(f"  Interpretation: {effect_size} effect size")
    
    print("\n6. SUCCESS RATE:")
    success_rate = np.mean(improvements > 0) * 100
    print(f"  Stage 2 better than Stage 1 in {success_rate:.1f}% of runs")
    
    return {
        't_statistic': t_stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'mean_improvement': np.mean(improvements),
        'success_rate': success_rate,
        'ci_low': ci_low,
        'ci_high': ci_high
    }


def visualize_results(results_df, stats_results):
    """
    Create visualizations for the multi-seed experiment results
    """
    print("\n" + "="*70)
    print("VISUALIZING RESULTS")
    print("="*70)
    
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Multi-Seed Experiment Results: Stage 1 vs Stage 2 Models', fontsize=16, fontweight='bold')
    
    ax1 = axes[0, 0]
    accuracy_data = pd.DataFrame({
        'Stage 1': results_df['Stage1_Accuracy'],
        'Stage 2': results_df['Stage2_Accuracy']
    })
    box = ax1.boxplot([accuracy_data['Stage 1'], accuracy_data['Stage 2']], 
                     labels=['Stage 1', 'Stage 2'])
    ax1.set_title('Accuracy Distribution Across Seeds')
    ax1.set_ylabel('Accuracy')
    ax1.grid(True, alpha=0.3)
    
    ax1.scatter([1, 2], [accuracy_data['Stage 1'].mean(), accuracy_data['Stage 2'].mean()], 
               color='red', zorder=3, label='Mean')
    ax1.legend()
    
    ax2 = axes[0, 1]
    seeds = range(1, len(results_df) + 1)
    ax2.plot(seeds, results_df['Improvement'], marker='o', linewidth=2, markersize=8)
    ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    ax2.fill_between(seeds, 0, results_df['Improvement'], 
                    where=results_df['Improvement']>0, alpha=0.3, color='green')
    ax2.fill_between(seeds, 0, results_df['Improvement'], 
                    where=results_df['Improvement']<0, alpha=0.3, color='red')
    ax2.set_title('Improvement (Stage 2 - Stage 1) Across Seeds')
    ax2.set_xlabel('Seed Number')
    ax2.set_ylabel('Improvement')
    ax2.grid(True, alpha=0.3)
    
    ax3 = axes[0, 2]
    ax3.scatter(results_df['Stage1_Accuracy'], results_df['Stage2_Accuracy'], 
               alpha=0.6, s=100)
    
    min_val = min(results_df['Stage1_Accuracy'].min(), results_df['Stage2_Accuracy'].min())
    max_val = max(results_df['Stage1_Accuracy'].max(), results_df['Stage2_Accuracy'].max())
    ax3.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label='y = x')
    
    ax3.set_title('Stage 1 vs Stage 2 Accuracy')
    ax3.set_xlabel('Stage 1 Accuracy')
    ax3.set_ylabel('Stage 2 Accuracy')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    ax4 = axes[1, 0]
    ax4.hist(results_df['Improvement'], bins=10, edgecolor='black', alpha=0.7)
    ax4.axvline(x=0, color='r', linestyle='--', alpha=0.5, label='Zero Improvement')
    ax4.axvline(x=results_df['Improvement'].mean(), color='g', linestyle='-', 
                alpha=0.7, label=f'Mean: {results_df["Improvement"].mean():.4f}')
    ax4.set_title('Distribution of Improvements')
    ax4.set_xlabel('Improvement (Stage 2 - Stage 1)')
    ax4.set_ylabel('Frequency')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    ax5 = axes[1, 1]
    success_count = (results_df['Improvement'] > 0).sum()
    failure_count = (results_df['Improvement'] <= 0).sum()
    bars = ax5.bar(['Stage 2 Better', 'Stage 2 Worse/Equal'], 
                  [success_count, failure_count], 
                  color=['green', 'red'], alpha=0.7)
    ax5.set_title(f'Success Rate: {success_count}/{len(results_df)} ({success_count/len(results_df)*100:.1f}%)')
    ax5.set_ylabel('Number of Seeds')
    
    for bar in bars:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{int(height)}', ha='center', va='bottom')
    
    ax6 = axes[1, 2]
    ax6.axis('off')
    
    summary_text = (
        f"STATISTICAL SUMMARY\n\n"
        f"Paired t-test:\n"
        f"  t-statistic = {stats_results['t_statistic']:.4f}\n"
        f"  p-value = {stats_results['p_value']:.6f}\n"
        f"  {'SIGNIFICANT' if stats_results['p_value'] < 0.05 else 'NOT SIGNIFICANT'}\n\n"
        f"Effect Size (Cohen's d):\n"
        f"  d = {stats_results['cohens_d']:.4f}\n\n"
        f"Mean Improvement:\n"
        f"  Δ = {stats_results['mean_improvement']:.4f}\n"
        f"  95% CI: [{stats_results['ci_low']:.4f}, {stats_results['ci_high']:.4f}]\n\n"
        f"Success Rate:\n"
        f"  {stats_results['success_rate']:.1f}% of seeds"
    )
    
    ax6.text(0.1, 0.9, summary_text, fontsize=10, fontfamily='monospace',
            verticalalignment='top', linespacing=1.5)
    
    plt.tight_layout()
    
    plt.savefig('/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/multi_seed_results.png', 
                dpi=300, bbox_inches='tight')
    plt.savefig('/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/multi_seed_results.pdf',
                bbox_inches='tight')
    print("Visualizations saved to multi_seed_results.png and multi_seed_results.pdf")
    
    plt.show()


def main():
    file_path = "/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/gcj2020.csv"
    
    print("Selecting configuration for multi-seed experiment...")
    print("="*70)
    print("OPTION 1: Top 1000 authors (fastest)")
    print("OPTION 2: Authors with ≥50 samples (balanced)")
    print("OPTION 3: Authors with ≥100 samples (more data per class)")
    print("="*70)
    
    choice = input("\nEnter your choice (1, 2, or 3): ")
    
    if choice == "1":
        min_samples = 2
        max_authors = 1000
        epochs_stage1 = 3
        epochs_stage2 = 5
    elif choice == "2":
        min_samples = 50
        max_authors = 100
        epochs_stage1 = 3
        epochs_stage2 = 3
    elif choice == "3":
        min_samples = 100
        max_authors = 50
        epochs_stage1 = 4
        epochs_stage2 = 3
    else:
        print("Invalid choice. Using Option 2.")
        min_samples = 50
        max_authors = 100
        epochs_stage1 = 3
        epochs_stage2 = 3
    
    print(f"\nRunning multi-seed experiment with {NUM_SEEDS} seeds...")
    print(f"Configuration: min_samples={min_samples}, max_authors={max_authors}")
    print(f"Stage 1 epochs: {epochs_stage1}, Stage 2 epochs: {epochs_stage2}")
    
    results = []
    
    for i, seed in enumerate(tqdm(SEEDS, desc="Running experiments")):
        stage1_acc, stage2_acc, improvement = run_single_experiment(
            seed=seed,
            file_path=file_path,
            min_samples=min_samples,
            max_authors=max_authors,
            epochs_stage1=epochs_stage1,
            epochs_stage2=epochs_stage2
        )
        
        results.append({
            'Seed': seed,
            'Stage1_Accuracy': stage1_acc,
            'Stage2_Accuracy': stage2_acc,
            'Improvement': improvement,
            'Improvement_Percent': (improvement / stage1_acc * 100) if stage1_acc > 0 else 0
        })
    
    results_df = pd.DataFrame(results)
    
    results_path = "/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/multi_seed_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"\nDetailed results saved to: {results_path}")
    
    print("\n" + "="*70)
    print("SUMMARY OF RESULTS ACROSS ALL SEEDS")
    print("="*70)
    print(results_df.to_string(index=False))
    
    stats_results = perform_statistical_analysis(results_df)
    
    visualize_results(results_df, stats_results)
    
    print("\n" + "="*70)
    print("FINAL CONCLUSION")
    print("="*70)
    
    mean_improvement = stats_results['mean_improvement']
    p_value = stats_results['p_value']
    success_rate = stats_results['success_rate']
    
    if p_value < 0.05:
        if mean_improvement > 0:
            print(f"✅ Stage 2 model is SIGNIFICANTLY BETTER than Stage 1 (p = {p_value:.6f})")
            print(f"   Average improvement: {mean_improvement:.4f} ({mean_improvement/stage1_acc*100:.1f}% relative)")
        else:
            print(f"❌ Stage 2 model is SIGNIFICANTLY WORSE than Stage 1 (p = {p_value:.6f})")
            print(f"   Average degradation: {abs(mean_improvement):.4f}")
    else:
        print(f"⚠️  No statistically significant difference between Stage 1 and Stage 2 (p = {p_value:.6f})")
    
    print(f"\nStage 2 performed better in {success_rate:.1f}% of the seeds")
    print(f"95% Confidence Interval for improvement: [{stats_results['ci_low']:.4f}, {stats_results['ci_high']:.4f}]")
    
    stats_df = pd.DataFrame([stats_results])
    stats_path = "/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/statistical_results.csv"
    stats_df.to_csv(stats_path, index=False)
    print(f"\nStatistical results saved to: {stats_path}")

if __name__ == "__main__":
    main()